# 5 · Digital gain and power meters

**Local Quartus workflow · host or container · one active FPGA design at a time**

`y=saturate(round(g·x)),  P=Σ(I²+Q²)/N`

Gain runs in FPGA logic/DSP alongside integer power and clipping meters. Twice the amplitude gives about +6.02 dB power until saturation.

Python model → native VDS diagram → real RTL/IP simulation → SOF → board → focused CyberEther page.

## Setup

Once, from this example folder: `bash scripts/install_local.sh --system` (Linux system packages need sudo), then select **Atum A3 IQ Demos** as the notebook kernel. Quartus/Questa/Nios tools must already be installed. Set the optional paths below if they are not on PATH. No Docker commands are needed.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 5
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)


## 1 · Python simulation

The code below is the editable numerical model. Its plots are synthetic/theory, separate from the RTL results and live hardware below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from notebook_plots import spectrum
from nios_fir import PROFILES, fixed_reference
from nios_ddc import reference, gain_reference
from iq_lab_samples import load_sample
raw, _ = load_sample("two-tone")
fs = 16e6
n = CHECKPOINT
t = np.arange(128)
z = 12000 * np.exp(2j * np.pi * t / 16)
a = np.rint(np.column_stack((z.real, z.imag))).astype(np.int16)
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
clips = []
powers = []
gains = [0.5, 1, 2, 4, 16]
for g in gains:
    out, m = gain_reference(a, int(g * 256), False)
    clips.append(m['gain_clips'])
    powers.append(m['post_power'] / len(a))
    if g in (1, 2, 4):
        axs[0].plot(out[:48, 0], label=f'{g:g}×')
axs[0].set(xlabel='Sample', ylabel='CI16 I', title='Gain + nearest rounding + saturation')
axs[0].legend()
axs[1].plot(gains, 10 * np.log10(np.array(powers) / powers[1]), 'o-')
axs[1].set(xscale='log', xlabel='Digital gain', ylabel='Mean power change (dB)', title='2× amplitude → about +6.02 dB')
axs[2].bar([str(g) + '×' for g in gains], clips)
axs[2].set(ylabel='Clipped complex samples', title='Clipping is counted in FPGA')
summary = {'2x_expected_power_change_dB': 20 * np.log10(2), 'gain_clip_counts': dict(zip(map(str, gains), clips))}
fig.suptitle(f"{CHECKPOINT} · Python simulation / integer reference")
fig.tight_layout()
plt.show()
summary


## 2 · Generate the actual FPGA block design

Rebuild the Nios initialization memory where present, recreate the native VDS system, validate and generate vendor IP. The PNG/SVG below comes from **Quartus-exported native connections**, rendered by Graphviz; it is not a hand-drawn or simulated topology. Clock/reset fanout is hidden to keep it readable.

In [ ]:
diagram_png = lesson.generate()
print("Exported:", diagram_png)

## 3 · RTL/IP simulation

Real gain/meter RTL: bypass, half, double, saturation and zero; all integer outputs and meters checked.

The plots below read the actual simulator output files. Assertions must pass before the result is plotted.

In [ ]:
simulation_result = lesson.simulate()
print("Simulation:", simulation_result["status"])
lesson.simulation_plot()

### Open the Questa GUI

Use a desktop terminal **on the machine running Quartus** (or its container desktop). Execute the printed commands. `wave.wlf` opens the saved real simulation; `view.do` adds its signals. To rerun interactively, start `vsim -gui` in the same directory and type `do gui.do` in the Transcript. This reruns the same assertions and leaves the waveform window open.

In [ ]:
lesson.questa_gui()

## 4 · Build the SOF

A source-matched, hash-verified cached bitfile can be reused for a quick presentation. Set `FORCE_REBUILD=True` to demonstrate a full Quartus compilation. A fresh checkout always requires a build.

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 5 · Program the physical board

This cell **programs the selected checkpoint**. It first stops a previous notebook-owned demo and releases UART/JTAG. Stop any unrelated acquisition service before running it; an unknown serial owner is refused. No ESP32 flashing or reset-button presses are needed with the installed suite.

In [ ]:
lesson.program()

## 6 · Focused CyberEther demo

Try 0.5×, 1×, 2×, then 4× or 16×. Watch actual power meters and clipping.

The page uses the bitfile just programmed. One board and one acquisition owner are shared across all notebooks.

In [ ]:
live = lesson.start_demo(host=HOST)
print(live)

Leave the page running while presenting. The next notebook stops it before programming. To stop it manually, run `stop_demo()` in the cell below. Live invites and captures stay in ignored runtime folders; do not publish their tokens.

In [ ]:
# stop_demo()